In [3]:
import json
from pyvis.network import Network
import networkx as nx

In [22]:
with open("/home/asagirova/arigraph/reproduction_cookbook/adaptive-pruning/graph_data.json",
    #f'/home/asagirova/arigraph/cookbook_extraction_prompt/graph_data.json',
   #'/home/asagirova/arigraph/reproduction_cookbook/reproduction_cookbook.json',
'r') as f:

    file = json.load(f)
triplets = file['triplets']


G = nx.Graph()  # Use Graph for undirected, DiGraph for directed

for s, o, r in triplets:
    G.add_edge(s, o, label=r['label'])

components = list(nx.connected_components(G))
#print(f"\nNumber of connected components: {len(components)}")
# Sort components by size (largest first)
components_sorted = sorted(components, key=len, reverse=True)

comps_to_join = set()
for idx in range(len(components_sorted)):
    component = components_sorted[idx]
    if len(component) <= 100:
        comps_to_join = comps_to_join.union(component)
        
    if idx == 10:
        break
if len(comps_to_join) > 0:
    subgraph = G.subgraph(comps_to_join).copy()
    vis_net(subgraph, '/home/asagirova/claude-code', save_as=f'old_graph_vis_{idx}')

/home/asagirova/claude-code/old_graph_vis_10.html


In [21]:
len(comps_to_join)

75

In [1]:
def vis_net(subgraph, exp_path, save_as=''):
    """
    Visualize graph with multiple connected components properly separated.
    Each component is positioned independently to prevent overlap.
    """
    import math
    # Find connected components within the subgraph
    components = list(nx.connected_components(subgraph))
    num_comps = len(components)
    
    # Create pyvis network
    net = Network(
        height="1000px",
        width="100%",
        bgcolor="#ffffff",
        font_color='#10000000' if num_comps < 2 else "black",
        directed=True,
    )

    import matplotlib.colors as mcolors
    colors = list(mcolors.TABLEAU_COLORS.values())
    if len(components) > len(colors):
        colors = list(mcolors.CSS4_COLORS.values())

    # Assign colors to nodes based on component
    component_map = {}
    for i, component in enumerate(components):
        color = colors[i % len(colors)]
        for node in component:
            component_map[node] = {'component_id': i, 'color': color}

    # Calculate layout for each component separately with proper spacing
    all_positions = {}
    #component_spacing = 2000  # Space between components
    
    # Arrange components in a grid layout
    #grid_cols = math.ceil(math.sqrt(len(components)))
    
    for idx, component in enumerate(components):
        # Create subgraph for this component
        comp_subgraph = subgraph.subgraph(component)
        
        pos = nx.spring_layout(comp_subgraph, k=1, iterations=50, scale=1000)
        
        
    
    # Add nodes with fixed positions
    for node in subgraph.nodes():
        
        net.add_node(
            node,
            label=node,
            title=node,
            color=component_map[node]['color'],
            shape='dot',
            size=10,
            
        )

    # Add edges
    for s, o, edge_data in subgraph.edges(data=True):
        net.add_edge(
            s, o, 
            label=edge_data.get('label', ''),
            title=edge_data.get('label', ''),
        )
    
    
    
    

    # Save and display
    filename = f'{exp_path}/{save_as}.html'
    net.show(filename, notebook=False)